<a href="https://colab.research.google.com/github/Gagangowdatg/BIS_1BM24CS098/blob/main/Ant%20Colony%20Optimisation%20for%20TSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# 1. DEFINE THE PROBLEM (24-Hour Microgrid Data)
hours = 24
# Simulated hourly electricity demand (kW)
demand = np.array([15, 14, 13, 13, 15, 20, 28, 35, 32, 28, 25, 24,
                   25, 26, 28, 32, 40, 48, 50, 45, 38, 30, 22, 18])

# Simulated Solar + Wind generation (kW)
solar = np.array([0, 0, 0, 0, 0, 2, 8, 18, 28, 35, 40, 42,
                  40, 36, 28, 18, 8, 2, 0, 0, 0, 0, 0, 0])
wind = np.array([12, 14, 15, 14, 12, 10, 8, 7, 6, 6, 8, 10,
                 11, 12, 14, 15, 16, 15, 14, 13, 14, 15, 14, 13])
renewable_gen = solar + wind

# Battery & Grid Constraints
BAT_MAX = 50.0       # Max battery capacity (kWh)
BAT_MIN = 10.0       # Min safe battery charge (kWh)
MAX_RATE = 15.0      # Max charge/discharge per hour (kW)
GRID_COST = 0.25     # $ per kWh bought from main grid
WASTE_PENALTY = 0.05 # Small penalty for wasting unused renewable energy

# Possible actions at each hour: -1 (Discharge), 0 (Idle), +1 (Charge)
actions = [-1, 0, 1]

# 2. INITIALIZE ACO PARAMETERS
num_ants = 30
max_iter = 100
alpha = 1.0   # Pheromone importance
beta = 2.0    # Heuristic importance
rho = 0.2     # Evaporation rate
Q = 100.0     # Pheromone deposit constant

# Pheromone matrix: shape (24 hours, 3 possible actions)
pheromone = np.ones((hours, len(actions)))

def evaluate_schedule(schedule):
    """Calculates total operating cost and enforces equipment constraints."""
    soc = 25.0  # Starting battery State of Charge (kWh)
    total_cost = 0.0

    for h in range(hours):
        net_power = renewable_gen[h] - demand[h] # Positive = surplus, Negative = deficit
        act = actions[schedule[h]]

        if act == 1: # CHARGE BATTERY
            # Can only charge up to MAX_RATE, available surplus, and battery capacity
            charge = min(MAX_RATE, max(0, net_power), BAT_MAX - soc)
            soc += charge
            unused_renewable = max(0, net_power - charge)
            grid_used = max(0, -net_power) # If deficit, still need grid

        elif act == -1: # DISCHARGE BATTERY
            # Can only discharge up to MAX_RATE, current deficit, and min safe limit
            deficit = max(0, -net_power)
            discharge = min(MAX_RATE, deficit, soc - BAT_MIN)
            soc -= discharge
            unused_renewable = max(0, net_power)
            grid_used = max(0, deficit - discharge)

        else: # IDLE
            unused_renewable = max(0, net_power)
            grid_used = max(0, -net_power)

        # Calculate cost for this hour
        total_cost += (grid_used * GRID_COST) + (unused_renewable * WASTE_PENALTY)

    return total_cost

def get_heuristic(hour):
    """Smart hint: prefer charging when surplus exists, discharging during deficit."""
    net = renewable_gen[hour] - demand[hour]
    if net > 0:
        return np.array([0.1, 0.3, 1.0]) # Favor Charge (+1)
    elif net < 0:
        return np.array([1.0, 0.3, 0.1]) # Favor Discharge (-1)
    return np.array([0.2, 1.0, 0.2])     # Favor Idle (0)

# 3-6. MAIN ACO ITERATIVE LOOP
best_schedule = None
best_cost = float('inf')

for iteration in range(max_iter):
    all_schedules = []
    all_costs = []

    # Step 3 & 4: Each ant constructs a 24-hour battery schedule
    for ant in range(num_ants):
        schedule = []
        for h in range(hours):
            tau = pheromone[h] ** alpha
            eta = get_heuristic(h) ** beta
            probs = (tau * eta) / np.sum(tau * eta)

            chosen_idx = np.random.choice(len(actions), p=probs)
            schedule.append(chosen_idx)

        cost = evaluate_schedule(schedule)
        all_schedules.append(schedule)
        all_costs.append(cost)

        if cost < best_cost:
            best_cost = cost
            best_schedule = schedule

    # Step 5: Update Pheromones (Evaporation + Deposit)
    pheromone *= (1 - rho)
    for schedule, cost in zip(all_schedules, all_costs):
        deposit = Q / (cost + 1e-5) # Lower cost = higher pheromone
        for h, act_idx in enumerate(schedule):
            pheromone[h, act_idx] += deposit

# 7. OUTPUT BEST SOLUTION
action_names = {-1: "Discharge", 0: "Idle     ", 1: "Charge   "}
print(f"Optimized Daily Operating Cost: ${best_cost:.2f}\n")
print("Hour | Demand | Renew | Net   | Battery Action")
print("-" * 48)
for h in range(hours):
    net = renewable_gen[h] - demand[h]
    act = action_names[actions[best_schedule[h]]]
    print(f" {h:02d}  | {demand[h]:4d}kW | {renewable_gen[h]:3d}kW | {net:+4d}kW | {act}")

Optimized Daily Operating Cost: $40.70

Hour | Demand | Renew | Net   | Battery Action
------------------------------------------------
 00  |   15kW |  12kW |   -3kW | Discharge
 01  |   14kW |  14kW |   +0kW | Idle     
 02  |   13kW |  15kW |   +2kW | Charge   
 03  |   13kW |  14kW |   +1kW | Charge   
 04  |   15kW |  12kW |   -3kW | Discharge
 05  |   20kW |  12kW |   -8kW | Idle     
 06  |   28kW |  16kW |  -12kW | Discharge
 07  |   35kW |  25kW |  -10kW | Discharge
 08  |   32kW |  34kW |   +2kW | Idle     
 09  |   28kW |  41kW |  +13kW | Charge   
 10  |   25kW |  48kW |  +23kW | Charge   
 11  |   24kW |  52kW |  +28kW | Charge   
 12  |   25kW |  51kW |  +26kW | Charge   
 13  |   26kW |  48kW |  +22kW | Charge   
 14  |   28kW |  42kW |  +14kW | Discharge
 15  |   32kW |  33kW |   +1kW | Charge   
 16  |   40kW |  24kW |  -16kW | Discharge
 17  |   48kW |  17kW |  -31kW | Discharge
 18  |   50kW |  14kW |  -36kW | Discharge
 19  |   45kW |  13kW |  -32kW | Discharge
 20 